In [ ]:
import numpy as np, matplotlib.pyplot as plt
from pyhnc import *

Set up grid, Ornstein-Zernike solvers and reference solvent.

In [ ]:
N = 2**15
L = 100
grid = Grid(L, N)
r, q = grid.r, grid.q

verbose = False

alpha = 0.5
niters = 1000
tol = 1e-12
solvent = Solver(grid, alpha=alpha, niters=niters, tol=tol)
solvent_rpa = RandomPhaseApproximationSolver(grid, alpha=alpha, niters=niters, tol=tol)

# DPD water solvent.
A00 = 25
ρ0 = 3.0
φ0 = potentials.DPD(A00)
solvent = solvent.solve(φ0, ρ0, monitor=verbose)
solvent_rpa = solvent_rpa.solve(φ0, ρ0, monitor=verbose)

solute = SoluteSolver(solvent)
solute_rpa = SoluteTestParticleRPA(solvent_rpa)

print(grid)
print(solvent)

plt.figure(figsize=(3.375, 3))
plt.plot(r, solvent.g, label='HNC')
plt.plot(r, solvent_rpa.g, label='RPA')
plt.legend(loc='best')
plt.xlabel('$r$')
plt.ylabel('$g(r)$')
plt.xlim([0, 3])
plt.ylim([0, 1.5])
plt.show()

In [ ]:
A01 = 10
A02 = 2*A01
A03 = 3*A01
φ01 = potentials.DPD(A01)
φ02 = potentials.DPD(A02)
φ03 = potentials.DPD(A03)

sol1 = solute.solve(φ01, monitor=verbose)
sol2 = solute.solve(φ02, monitor=verbose)
sol3 = solute.solve(φ03, monitor=verbose)
sol2_rpa = solute_rpa.solve(φ02, monitor=verbose)

C = solvent.pressure
# C = ρ0
ψ1q = -sol1.hq * np.sqrt(ρ0 /(C * solvent.Sq))
ψ1 = grid.fourier_bessel_backward(ψ1q)
ψ2q = -sol2.hq * np.sqrt(ρ0 /(C * solvent.Sq))
ψ2 = grid.fourier_bessel_backward(ψ2q)
ψ3q = -sol3.hq * np.sqrt(ρ0 /(C * solvent.Sq))
ψ3 = grid.fourier_bessel_backward(ψ3q)

ψ2_additive = 2*ψ1 - ψ1**2
ψ2q_additive = grid.fourier_bessel_forward(ψ2_additive)
h02q_additive = - ψ2q_additive / np.sqrt(ρ0 /(C * solvent.Sq))
h02_additive = grid.fourier_bessel_backward(h02q_additive)
g02_additive = 1 + h02_additive

ψ3_additive = ψ1 + ψ2 - ψ1*ψ2
ψ3q_additive = grid.fourier_bessel_forward(ψ3_additive)
h03q_additive = - ψ3q_additive / np.sqrt(ρ0 /(C * solvent.Sq))
h03_additive = grid.fourier_bessel_backward(h03q_additive)
g03_additive = 1 + h03_additive

plt.figure(figsize=(3.375, 3.375))
# pl0, = plt.plot(r, solvent.g, lw=0.5, label=f'$A_{{00}} = {A00}$')
pl1, = plt.plot(r, sol1.g, lw=0.5, label=f'$A_{{01}} = {A01}$')
pl2, = plt.plot(r, sol2.g, lw=0.5, label=f'$A_{{02}} = 2A_{{01}} = {A02}$')
pl3, = plt.plot(r, sol3.g, lw=0.5, label=f'$A_{{03}} = 3A_{{01}} = {A03}$')

plt.plot(r, g02_additive, '--', lw=0.5, c=pl2.get_color(),
         label=r'$\psi_2 \approx 2 \psi_1 - \psi_1^2$')
plt.plot(r, g03_additive, '--', lw=0.5, c=pl3.get_color(),
         label=r'$\psi_3 \approx \psi_1 + \psi_2 - \psi_1 \psi_2$')

# φ02q = grid.fourier_bessel_forward(φ02(r))
# h02_rpa = grid.fourier_bessel_backward(-φ02q / (1 + ρ0*φ02q))
# plt.plot(r, 1+h02_rpa, ':', lw=0.5, c=pl2.get_color(), label=r'RPA')
# φ03q = grid.fourier_bessel_forward(φ03(r))
# h03_rpa = grid.fourier_bessel_backward(-φ03q / (1 + ρ0*φ03q))
# plt.plot(r, 1+h03_rpa, ':', lw=0.5, c=pl3.get_color())

plt.axhline(y=0)
plt.legend(loc='best')
plt.xlim([0, 2])
plt.ylim([-0.1, 1.3])
plt.xlabel('$r$')
plt.ylabel('$g_{0i}(r)$')
plt.savefig('g0i_additive.pdf')
plt.savefig('g0i_additive.png')
plt.xlim([0.7, 2.5])
plt.ylim([0.9, 1.25])
plt.savefig('g0i_additive_zoomed.pdf')
plt.savefig('g0i_additive_zoomed.png')
plt.show()

Show the shape of the generalisd indicators, extracted from one of two methods.

In [ ]:
F1 = (0.5 * sol1.h * (sol1.h - sol1.c) - sol1.c) * ρ0
F2 = (0.5 * sol2.h * (sol2.h - sol2.c) - sol2.c) * ρ0
F3 = (0.5 * sol3.h * (sol3.h - sol3.c) - sol3.c) * ρ0
ψ1_approx = F1 / solvent.pressure
ψ2_approx = F2 / solvent.pressure
ψ3_approx = F3 / solvent.pressure
ψ2_additive_approx = 2*ψ1_approx - ψ1_approx**2
ψ3_additive_approx = ψ1_approx + ψ2_approx - ψ1_approx*ψ2_approx

fig, (ax1, ax2) = plt.subplots(nrows=2, figsize=(3.375, 3.375), sharex=True)

pl1, = ax1.plot(r, ψ1, label=f'$\psi_1$ ($A_{{01}} = {A01}$)')
ax2.plot(r, ψ1_approx, '-', c=pl1.get_color(),
         label=r'$\psi_1 = \frac{\rho_0}{p} \left( \frac{h_{10}(h_{10}-c_{10})}{2} - c_{10} \right)$')
pl2, = ax1.plot(r, ψ2, label=f'$\psi_2$ ($A_{{02}} = 2A_{{01}} = {A02}$)')
ax2.plot(r, ψ2_approx, '-', c=pl2.get_color(),
         label=r'$\psi_2 = \frac{\rho_0}{p} \left( \frac{h_{20}(h_{20}-c_{20})}{2} - c_{20} \right)$')
pl3, = ax1.plot(r, ψ3, label=f'$\psi_3$ ($A_{{03}} = 3A_{{01}} = {A03}$)')
ax2.plot(r, ψ3_approx, '-', c=pl3.get_color(),
         label=r'$\psi_3 = \frac{\rho_0}{p} \left( \frac{h_{20}(h_{20}-c_{20})}{2} - c_{20} \right)$')

ax1.plot(r, ψ2_additive, '--', c=pl2.get_color(),
         label=r'$\psi_2 \approx 2 \psi_1 - \left(\psi_1\right)^2$')
ax2.plot(r, ψ2_additive_approx, '--', c=pl2.get_color(),
         label=r'$\psi_2 \approx 2 \psi_1 - \left(\psi_1\right)^2$')
ax1.plot(r, ψ3_additive, '--', c=pl3.get_color(),
         label=r'$\psi_3 \approx \psi_1 + \psi_2 - \psi_1 \psi_2$')
ax2.plot(r, ψ3_additive_approx, '--', c=pl3.get_color(),
         label=r'$\psi_3 \approx \psi_1 + \psi_2 - \psi_1 \psi_2$')



ax1.text(0.99, 0.975, r'\begin{center}additive $W_{ij}$ \quad $\widehat\psi_i = -\sqrt{\frac{\rho_0}{\beta p \, S_{00}}} \widehat{h}_{i0}$\end{center}', ha='right', va='top', transform=ax1.transAxes)
ax2.text(0.99, 0.975, r'\begin{center}additive $\Delta \Omega_{ij}$\\$\psi_i = \frac{\rho_0}{p} \left( \frac{h_{i0}(h_{i0}-c_{i0})}{2} - c_{i0} \right)$\end{center}', ha='right', va='top', transform=ax2.transAxes)

ax1.legend(loc='best')
# ax2.legend(loc='best')
plt.xlim([0, 2])
plt.xlabel('$r$')
for ax in [ax1, ax2]: ax.set_ylabel(r'$\psi_i$')

for letter, ax in zip('ab', [ax1, ax2]):
    t = ax.text(0.04, 0.025, rf'\textbf{{{letter}}}', fontsize=18,
                transform=ax.transAxes, ha='left', va='bottom')
    t.set_in_layout(False)

plt.savefig('indicators.pdf')
plt.savefig('indicators.png')
plt.show()

Let's look at the effective radius implied by each quasi-indicator $\psi_i$. First, the average radius is
$$
\langle r \rangle = \frac{\int \mathrm{d}r \, r^3 \psi(r)}{\int \mathrm{d}r \, r^2 \psi(r)}\,.
$$
We can also define an effective volume
$$
V = 4 \pi \int \mathrm{d}r \, r^2 \psi(r)\,,
$$
and an equivalent hard-sphere radius
$$
R = \left( \frac{3 V}{4\pi} \right)^{1/3}\,.
$$
We calculate these below, for the indicators extracted from each method (indicated by subscript).

In [ ]:
from scipy.integrate import simpson

print('A0i   <r>₁   <r>₂     V₁     V₂   <R>₁   <R>₂')
for A0i, ψ, ψ_approx in [(10, ψ1, ψ1_approx), (20, ψ2, ψ2_approx), (30, ψ3, ψ3_approx)]:
    r1 = simpson(r**3 * ψ_approx, r) / simpson(r**2 * ψ_approx, r)
    r2 = simpson(r**3 * ψ, r) / simpson(r**2 * ψ, r)
    V1 = simpson(4*np.pi * r**2 * ψ_approx, r)
    V2 = simpson(4*np.pi * r**2 * ψ, r)
    R1 = (3*V1/(4*np.pi))**(1/3)
    R2 = (3*V2/(4*np.pi))**(1/3)
    print(f' {A0i} {r1:.4f} {r2:.4f} {V1:.4f} {V2:.4f} {R1:.4f} {R2:.4f}')

In [ ]:
from ipywidgets import interact
# from scipy.integrate import cumulative_simpson
from scipy.integrate import solve_ivp
from scipy.interpolate import CubicSpline
from scipy.differentiate import derivative

p = solvent.pressure

def indicator(sol, γ, κ1=0., κ2=0.):
    F = (0.5 * sol.h * (sol.h - sol.c) - sol.c) * ρ0 # RHS / forcing term
    F = CubicSpline(r, F)

    # IF1 = np.exp(p/γ * r) # integrating factor
    # integrand = np.concatenate([[0], IF1 * F1])
    # domain = np.concatenate([[0], r])
    # integral = cumulative_simpson(integrand, x=domain, initial=0)[1:]
    # χ1 = (χ0 + integral) / IF1
    # χ1p = p/γ * (ψ1_approx - χ1)

    if γ < 0:

        # if p*χ0 > F1[0]:
        def rhs(r, χ):
            return (p * χ - F(r)) / (γ + 2*κ1/r + κ2/r**2)

        χ0 = F(r[0]) / p
        χ = solve_ivp(
            rhs, (r[0], r[-1]), [χ0], t_eval=r,
            method='LSODA', rtol=1e-4, atol=1e-6,
        ).y[0]
        χp = -(F(r) - p * χ) / (γ + 2*κ1/r + κ2/r**2)

    elif γ == 0.:
        χ = F(r) / p
        χp = derivative(F, r).df / p
    else:
        raise ValueError

    return χ, χp

def plot_psi(γ=0):
    global p, sol1, sol2, sol3

    ψ1q = -sol1.hq * np.sqrt(ρ0 /(p * solvent.Sq))
    ψ1 = grid.fourier_bessel_backward(ψ1q)
    ψ2q = -sol2.hq * np.sqrt(ρ0 /(p * solvent.Sq))
    ψ2 = grid.fourier_bessel_backward(ψ2q)
    ψ3q = -sol3.hq * np.sqrt(ρ0 /(p * solvent.Sq))
    ψ3 = grid.fourier_bessel_backward(ψ3q)

    ψ3_additive = ψ1 + ψ2 - ψ1*ψ2
    ψ3q_additive = grid.fourier_bessel_forward(ψ3_additive)
    h03q_additive = - ψ3q_additive / np.sqrt(ρ0 /(p * solvent.Sq))
    h03_additive = grid.fourier_bessel_backward(h03q_additive)
    g03_additive = 1 + h03_additive
    W03_additive = -np.log(g03_additive * np.exp(sol3.potential(r)))

    χ1, χ1p = indicator(sol1, γ)
    χ2, χ2p = indicator(sol2, γ)
    χ3, χ3p = indicator(sol3, γ)
    χ1q = grid.fourier_bessel_forward(χ1)
    χ2q = grid.fourier_bessel_forward(χ2)
    χ1pq = grid.fourier_bessel_forward(χ1p)
    χ2pq = grid.fourier_bessel_forward(χ2p)
    W30q_additive = -p * χ1q*χ2q + γ*(χ1q*χ2pq + χ1pq*χ2q)
    W30_additive = grid.fourier_bessel_forward(W30q_additive)

    fig, (ax1, ax2) = plt.subplots(nrows=2, figsize=(3.375, 3.375), sharex=True)

    pl1, = ax1.plot(r, χ1, '-', label=r'$\chi_1$')
    pl2, = ax1.plot(r, χ2, '-', label=r'$\chi_2$')
    pl3, = ax1.plot(r, χ3, '-', label=r'$\chi_3$')
    ax1.plot(r, χ1p, '--', c=pl1.get_color(), label=r"$\chi_1'$")
    ax1.plot(r, χ2p, '--', c=pl2.get_color(), label=r"$\chi_2'$")
    ax1.plot(r, χ3p, '--', c=pl3.get_color(), label=r"$\chi_3'$")

    # if γ != 0: f = (p*χ**2 - 2*γ*χ*χp)
    # else: f = p*χ**2
    # ax2.plot(r, f, '-', label=r"$p \chi^2 - 2\gamma \chi \chi'$")
    # plt.plot(r, ψ**2, ':', label=r'$|\psi|^2$')
    W10 = -np.log(sol1.g * np.exp(sol1.potential(r)))
    W20 = -np.log(sol2.g * np.exp(sol2.potential(r)))
    W30 = -np.log(sol3.g * np.exp(sol3.potential(r)))
    ax2.plot(r, W10, c=pl1.get_color(), label='$W_{10}$ HNC')
    ax2.plot(r, W20, c=pl2.get_color(), label='$W_{20}$ HNC')
    ax2.plot(r, W30, c=pl3.get_color(), label='$W_{30}$ HNC')
    ax2.plot(r, W30_additive, '--', c=pl3.get_color(), label='$W_{30}$ additive')
    ax2.plot(r, W03_additive, ':', c=pl3.get_color(), label='$W_{30}$ additive (2)')

    ax1.legend(ncols=2, loc='lower right')
    ax2.legend(loc='best')
    plt.xlim([0, 3])
    ax2.set_xlabel('$r$')
    ax1.set_ylabel(r'$\chi$')
    ax2.set_ylabel(r'$W_{i0}$')
    plt.show()

interact(plot_psi, γ=(-50, 0, 1e-4))